### 使用 OpenAI API 实现多步、有依赖关系的函数调用（Chained Function Calling）
https://platform.openai.com/docs/guides/function-calling#handling-function-calls

它生动地展示了如何构建一个能够自主规划并分步解决复杂问题的 AI 助手。具体来说，AI 首先识别出要查询天气，必须先获取地理坐标，于是它按顺序调用了两个不同的外部工具：`get_coordinates` 和 `get_weather`。

整个过程通过一个循环和持续更新的对话历史来管理，完美模拟了 AI 进行逻辑推理并与外部世界交互的完整流程，是构建高级 AI Agent 的一个经典范例。

In [1]:
from openai import OpenAI
import json
from dotenv import load_dotenv
import requests

load_dotenv()
client = OpenAI()

tools = [{
  "type": "function",
  "function": {
    "name": "get_coordinates",
    "description": "Retrieves the coordinates for the given location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "City and country e.g. Wuhan"
        }
      },
      "required": [
        "location"
      ],
      "additionalProperties": False
    },
    "strict": True
  }
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

In [ ]:
def get_coordinates(location):
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json")
    data = response.json()
    
    # Extract latitude and longitude from the response
    if "results" in data and len(data["results"]) > 0:
        latitude = data["results"][0]["latitude"]
        longitude = data["results"][0]["longitude"]
        # return latitude, longitude
        return json.dumps({"latitude": latitude, "longitude": longitude})
    else:
        raise ValueError("Location not found in the API response")

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return json.dumps(data.get("current", {"error": "Weather data not found"}))

In [ ]:
available_functions = {
    "get_coordinates": get_coordinates,
    "get_weather": get_weather,
}
messages = [{"role": "user", "content": "What's the weather like in Wuhan today? You must invoke all the tools necessary to answer this question."}]

In [2]:
while True:
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    response_message = completion.choices[0].message
    print("response_message = ", response_message)
    
    # 检查模型是否决定调用工具
    if response_message.tool_calls:
        print("模型决定调用工具...")
        
        # 将模型的回复（包含工具调用请求）添加到对话历史中
        messages.append(response_message)
        
        # 遍历并执行所有工具调用
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            
            # 调用本地函数
            function_response = function_to_call(**function_args)
            
            # 将函数执行的结果添加到对话历史中
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            })
            print(f"已将函数 '{function_name}' 的结果返回给模型。")
            print("----------------------------------------------------")
            print("下一步: 将函数结果发回模型，让它继续...")
            print("----------------------------------------------------")
            
    else:
        # 如果模型不再调用工具，说明它已准备好生成最终答案
        print("所有工具已执行完毕，模型正在生成最终回答...")
        print("----------------------------------------------------")
        print("最终回答:")
        print(response_message.content)
        break # 结束循环

response_message =  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_KNC74TPgXadx12qLv7ZPLgem', function=Function(arguments='{"location":"Wuhan"}', name='get_coordinates'), type='function')])
模型决定调用工具...
已将函数 'get_coordinates' 的结果返回给模型。
----------------------------------------------------
下一步: 将函数结果发回模型，让它继续...
----------------------------------------------------
response_message =  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_jk8wo12r3GQ0YiuTXkMWRbQN', function=Function(arguments='{"latitude":30.58333,"longitude":114.26667}', name='get_weather'), type='function')])
模型决定调用工具...
已将函数 'get_weather' 的结果返回给模型。
----------------------------------------------------
下一步: 将函数结果发回模型，让它继续...
----------------------------------------------------
response_message 